# Forex Strategy Optimizer

This notebook provides a Meta Trader-style strategy tester that exhaustively tests all possible combinations of filters.

## How it works

1. **Filter Dimensions**: Each column from the CSV (EMA, BOS/CH, SL, News, Hour, etc.) becomes a "dimension"
2. **Combinations**: The optimizer generates all possible combinations of these filters
3. **Backtesting**: Each combination is backtested across all RRR ratios (1:1, 1:2, 1:3)
4. **Results**: Strategies are ranked by edge (profitability above breakeven)

## Parameters

- **max_filters**: Maximum number of filters per strategy (e.g., 3 = up to 3 conditions)
- **min_trades**: Minimum trade count required (e.g., 10 trades minimum)
- **min_edge**: Minimum edge percentage required (e.g., 5.0 = 5% above breakeven)
- **top_n**: Show only top N strategies (or None for all)

In [1]:
# Import required libraries
import sys
import importlib

# Add parent directory to path
sys.path.insert(0, '..')

# Enable automatic reloading
%load_ext autoreload
%autoreload 2

# Import utilities
from utils import load_and_clean_data
from utils.optimizer import (
    optimize_strategies,
    display_optimization_results,
    export_optimization_results
)

# Load the data
print("Loading data...")
df = load_and_clean_data()
print(f"Loaded {len(df)} trades")

Loading data...
Loaded 1102 trades


## Quick Test: Top 50 Strategies (Max 2 Filters)

Start with a quick test to see the best double-setup strategies.

In [2]:
# Quick optimization: max 2 filters, top 50 results
results = optimize_strategies(
    df,
    max_filters=2,         # Test up to 2 filters per strategy
    min_filters=1,         # Include single-filter strategies too
    min_trades=100,        # Require at least 100 trades
    min_edge=0.0,          # Show all strategies (even negative edge)
    rrr_ratios=[1, 2, 3],  # Test all RRR ratios
    top_n=None             # Unlimited list
)

display_optimization_results(results, title="Top 50 Strategies (Max 2 Filters)")

Generating strategy combinations (max 2 filters)...
Total combinations to test: 555

Backtesting 555 strategies...
  Processed 500/555 strategies...

Optimization complete! Found 182 strategies meeting criteria.


,Strategy,Trades,RRR,Win %,Edge,Profit Factor,Expected Payoff,Drawdown %
1,Hour=10-12 + SL=5-10,128,1:2,44.5%,11.2%,1.61,0.34R,55.5%
2,Hour=10-12 + SL=5-10,128,1:3,32.8%,7.8%,1.47,0.31R,67.2%
3,EMA=Aligned + Weekday=Tuesday,140,1:2,40.0%,6.7%,1.33,0.20R,60.0%
4,News=News >2h + SL=5-10,123,1:2,39.8%,6.5%,1.32,0.20R,60.2%
5,Direction=Buy + SL=5-10,235,1:2,39.6%,6.3%,1.31,0.19R,60.4%
6,Direction=Buy + Hour=10-12,153,1:2,39.2%,5.9%,1.29,0.18R,60.8%
7,Hour=15,117,1:3,30.8%,5.8%,1.33,0.23R,69.2%
8,Direction=Buy + Weekday=Tuesday,118,1:2,39.0%,5.7%,1.28,0.17R,61.0%
9,30M Trend=Aligned + Weekday=Tuesday,131,1:2,38.9%,5.6%,1.27,0.17R,61.1%
10,30M Trend=Aligned + Weekday=Thursday,121,1:3,30.6%,5.6%,1.32,0.22R,69.4%


## Full Optimization: All 3-Filter Combinations

Run a comprehensive test with up to 3 filters per strategy. This will test thousands of combinations.

In [3]:
# Full optimization: max 3 filters, top 100 results
results = optimize_strategies(
    df,
    max_filters=3,         # Test up to 3 filters per strategy
    min_filters=1,         # Include single-filter strategies
    min_trades=200,        # Require at least 100 trades
    min_edge=5.0,          # Only show strategies with 5%+ edge
    rrr_ratios=[1, 2, 3],  # Test all RRR ratios
    top_n=None             # Unlimited
)

display_optimization_results(results, title="Top 100 Strategies (Max 3 Filters, 5%+ Edge)")

Generating strategy combinations (max 3 filters)...
Total combinations to test: 4495

Backtesting 4495 strategies...
  Processed 500/4495 strategies...
  Processed 1000/4495 strategies...
  Processed 1500/4495 strategies...
  Processed 2000/4495 strategies...
  Processed 2500/4495 strategies...
  Processed 3000/4495 strategies...
  Processed 3500/4495 strategies...
  Processed 4000/4495 strategies...

Optimization complete! Found 2 strategies meeting criteria.


,Strategy,Trades,RRR,Win %,Edge,Profit Factor,Expected Payoff,Drawdown %
1,Direction=Buy + SL=5-10,235,1:2,39.6%,6.3%,1.31,0.19R,60.4%
2,BOS/CH=BOS + EMA=Aligned + SL=5-10,227,1:2,38.8%,5.5%,1.27,0.16R,61.2%


## Aggressive Optimization: 4+ Filters

Test complex strategies with 4+ filters. Warning: This can take longer and may overfit to the data.

In [4]:
# Aggressive optimization: max 4 filters
results = optimize_strategies(
    df,
    max_filters=4,      # Test up to 4 filters per strategy
    min_filters=3,      # Only show 3+ filter strategies
    min_trades=5,       # Lower minimum (complex filters = fewer trades)
    min_edge=10.0,      # High edge threshold (10%+)
    rrr_ratios=[1, 2, 3],
    top_n=50
)

display_optimization_results(results, title="Top 50 Complex Strategies (3-4 Filters, 10%+ Edge)")

Generating strategy combinations (max 4 filters)...
Total combinations to test: 21895

Backtesting 21895 strategies...
  Processed 500/21895 strategies...
  Processed 1000/21895 strategies...
  Processed 1500/21895 strategies...
  Processed 2000/21895 strategies...
  Processed 2500/21895 strategies...
  Processed 3000/21895 strategies...
  Processed 3500/21895 strategies...
  Processed 4000/21895 strategies...
  Processed 4500/21895 strategies...
  Processed 5000/21895 strategies...
  Processed 5500/21895 strategies...
  Processed 6000/21895 strategies...
  Processed 6500/21895 strategies...
  Processed 7000/21895 strategies...
  Processed 7500/21895 strategies...
  Processed 8000/21895 strategies...
  Processed 8500/21895 strategies...
  Processed 9000/21895 strategies...
  Processed 9500/21895 strategies...
  Processed 10000/21895 strategies...
  Processed 10500/21895 strategies...
  Processed 11000/21895 strategies...
  Processed 11500/21895 strategies...
  Processed 12000/21895 str

,Strategy,Trades,RRR,Win %,Edge,Profit Factor,Expected Payoff,Drawdown %
1,Hour=11 + News=News >2h + SL=5-10 + Weekday=Tuesday,6,1:2,100.0%,66.7%,∞,2.00R,0.0%
2,Hour=11 + News=With News + SL=5-10 + Weekday=Tuesday,6,1:2,100.0%,66.7%,∞,2.00R,0.0%
3,EMA=Counter + Hour=10 + Weekday=Friday,7,1:3,85.7%,60.7%,18.00,2.43R,14.3%
4,EMA=Counter + Hour=10 + News=News >2h + Weekday=Friday,7,1:3,85.7%,60.7%,18.00,2.43R,14.3%
5,EMA=Counter + Hour=10 + News=With News + Weekday=Friday,7,1:3,85.7%,60.7%,18.00,2.43R,14.3%
6,EMA=Counter + Hour=10 + SL=≤10 + Weekday=Friday,7,1:3,85.7%,60.7%,18.00,2.43R,14.3%
7,EMA=Counter + Hour=10 + SL=≤15 + Weekday=Friday,7,1:3,85.7%,60.7%,18.00,2.43R,14.3%
8,Direction=Sell + Hour=18 + SL=5-10 + Weekday=Monday,6,1:3,83.3%,58.3%,15.00,2.33R,16.7%
9,BOS/CH=CH + Hour=10 + News=With News + Weekday=Friday,6,1:3,83.3%,58.3%,15.00,2.33R,16.7%
10,BOS/CH=BOS + EMA=Aligned + SL=>15 + Weekday=Thursday,6,1:3,83.3%,58.3%,15.00,2.33R,16.7%


## Custom Optimization

Customize your own optimization parameters below:

In [6]:
# Custom optimization - adjust parameters as needed
custom_results = optimize_strategies(
    df,
    max_filters=3,           # Your choice: 1, 2, 3, 4, etc.
    min_filters=1,           # Your choice: 1, 2, 3, etc.
    min_trades=400,          # Your choice: adjust based on data size
    min_edge=0.0,            # Your choice: 0 for all, 5 for 5%+, etc.
    rrr_ratios=[3],          # 1:3 RRR
    top_n=None               # Unlimited
)

display_optimization_results(custom_results, title="Custom Optimization Results")

Generating strategy combinations (max 3 filters)...
Total combinations to test: 4495

Backtesting 4495 strategies...
  Processed 500/4495 strategies...
  Processed 1000/4495 strategies...
  Processed 1500/4495 strategies...
  Processed 2000/4495 strategies...
  Processed 2500/4495 strategies...
  Processed 3000/4495 strategies...
  Processed 3500/4495 strategies...
  Processed 4000/4495 strategies...

Optimization complete! Found 26 strategies meeting criteria.


,Strategy,Trades,RRR,Win %,Edge,Profit Factor,Expected Payoff,Drawdown %
1,30M Trend=Aligned + BOS/CH=BOS + EMA=Aligned,409,1:3,28.1%,3.1%,1.17,0.12R,71.9%
2,30M Trend=Aligned + BOS/CH=BOS + SL=≤10,417,1:3,28.1%,3.1%,1.17,0.12R,71.9%
3,SL=5-10,440,1:3,28.0%,3.0%,1.16,0.12R,72.0%
4,30M Trend=Aligned + BOS/CH=BOS,466,1:3,27.5%,2.5%,1.14,0.10R,72.5%
5,30M Trend=Aligned + BOS/CH=BOS + SL=≤15,446,1:3,27.4%,2.4%,1.13,0.09R,72.6%
6,30M Trend=Aligned + EMA=Aligned + SL=≤10,426,1:3,27.2%,2.2%,1.12,0.09R,72.8%
7,BOS/CH=BOS + EMA=Aligned + SL=≤10,528,1:3,27.1%,2.1%,1.11,0.08R,72.9%
8,30M Trend=Aligned + SL=≤10,554,1:3,26.9%,1.9%,1.10,0.08R,73.1%
9,30M Trend=Aligned + EMA=Aligned,481,1:3,26.6%,1.6%,1.09,0.06R,73.4%
10,30M Trend=Aligned + EMA=Aligned + SL=≤15,459,1:3,26.6%,1.6%,1.09,0.06R,73.4%
